# E0b: is B worth anything when A differs only by a monotone output map?

## Description

<!-- Written before the first run; kept up to date. -->

**Question.** A and B share the search space *and* the function; they differ only by a monotone output map g of [0, 1] onto itself. This is the case of a metric whose scale or saturation differs between datasets (floor and ceiling effects; `docs/hpo_task_distortions.md` §9). **How much better does a Bayes learner predict A's held-out points than A alone does?** The learner knows the pair prior, but not g and not whether B is related at all. **And how much of the known-g headroom survives having to infer g?**
Unlike E0 (#1), nothing about x has to be inferred. So this isolates the identification of an *output* relation. With only 2 parameters, an exact reference is cheap, and there is no sampler question (cf. notebook 02).

**Setup.**
- **Pairs** (`experimental/outmap.py`). One latent f is drawn from pfn-repro's BNN prior (the frozen PFN's training prior, ECDF scale).
  - B observes f directly at n_B uniform points.
  - A observes y_A = g(f(x) + noise) at n_A nested uniform points, plus 32 held-out queries.
  - g is a Kumaraswamy CDF 1 − (1 − u^α)^β with log α, log β ~ N(0, 0.5²). It is monotone and maps [0, 1] onto itself, so A's y stay on the PFN's support. Affine maps would leave it: the PFN embeds raw y and doesn't normalize per context.
  - The prior adds its noise before the ECDF, so g⁻¹(y_A) is exactly a prior observation of f.

  The strata are:
  - `identity`: g = id (sanity: B is A's own task);
  - `outmap`: g drawn from the family;
  - `unrelated`: B observes an independent function; g = id or drawn from the family, with probability ½ each (negative transfer).
- **Sizes scale with d.** d ∈ {1, 2}, n_B = 32 / 64, and n_A ∈ {0, d, 2d, 4d, 8d}. The largest context is 80, inside pfn_large's n ≤ 100.
- **Predictors**, all with the frozen `pfn_large`, exact by change of variables, log p(y_A) = log PFN(g⁻¹(y_A)) + log |(g⁻¹)'(y_A)|:
  - `lower_bayes`: A alone, with g inferred from A (a mixture over g). This is **the baseline: A alone under the same pair prior.**
  - `lower`: plain PFN(A), which treats A as an untransformed prior draw. It shows what ignoring g costs without B.
  - `naive`: PFN(B ∪ A), pooling without the output map.
  - `oracle_g`: PFN(B ∪ g⁻¹(A)) with the true g (upper bound).
  - `bayes`: the posterior predictive of the learner whose prior is the pair prior. It is a mixture over {B used, B ignored} × {g = id, g drawn by adaptive multiple importance sampling}, weighted by prior mass × the PFN chain-rule likelihood of A's points. The kind prior is ⅓ each.
  - **The g sampler** (`_MapPosterior`): a randomized 16 × 16 quantile grid of the prior, then 4 rounds of 24 draws per n_A from Gaussians fitted to the current posterior. All draws are weighted against the mixture of all proposals (balance heuristic). A fixed grid collapsed (ESS 1–3, MC error 0.02–0.05) in a scratch check: with dense B, the g posterior is very narrow at this noise level.
- **Monte-Carlo error.** On 10 pairs per cell, `bayes` is recomputed with independent sampler draws.
- **Grid and seeds.** 6 cells: d ∈ {1, 2} × kind ∈ {identity, outmap, unrelated}. Pilot: 50 pairs per cell (pair split 0); d = 1 runs on the laptop CPU, d = 2 on ulysses (GPU), as two sweeps at one commit; confirm: fresh pairs (split 1). Pairs are identified by (split, d, kind, i), with seeds disjoint from E0's.

**What to look for.** Gains over `lower_bayes` in nats per held-out query, with 95% pair-bootstrap CIs, against n_A / d:
- `gain_bayes` (what B is worth), `gain_oracle` (g known), G = gain_bayes / gain_oracle;
- **n_A = 0, the in-context prior**: what B alone says about A before any A point, weighted by the prior's relatedness. This gets no rule; it is a number to read (working hypothesis H3);
- `gain_naive` (what ignoring g costs when pooling) and `lower` vs `lower_bayes` (the same, without B);
- diagnostics: posterior mass per kind, the posterior RMS error of g over [0, 1], the ESS of the g grid, and the MC error.

**Decision rule (fixed before the pilot; `analysis/outmap.py::verdict`; drafted by Claude 2026-10-22, not edited by the user before the run).** Gate cells: `outmap` × d ∈ {1, 2} × n_A ∈ {2d, 4d, 8d}.
- **INVALID** unless the Bayes learner recovers ≥ 90% of the oracle gain on identity pairs at n_A = 8d, and the oracle gain is > 0 in every gate cell.
- **IMPRECISE** if the MC error exceeds 0.025 nats in any gate cell (then a finer grid first).
- **IDENTIFIED** if in every gate cell the CI lower bound of gain_bayes is > 0.10 nats **and** G > 0.75.
- **NONE** if in every gate cell the CI upper bound of gain_bayes is < 0.05 nats.
- **PARTIAL** otherwise.
- *Safety (reported with the verdict).* On unrelated pairs (n_A ≥ 1), the CI lower bound of gain_bayes should stay above −0.05.

What each verdict means:
- **IDENTIFIED**: an output relation is cheap to infer. This stratum becomes E1's first test of whether a Meta-PFN identifies a relation, with an exact ceiling.
- **PARTIAL**: inferring even a 2-parameter output relation costs a real share. That share is reported as the price, and it bounds what a Meta-PFN can reach.
- **NONE**: B is useless even with x shared. That contradicts H3 and needs a diagnosis before anything else.

**Limits.**
- **Noise before g**, which makes the reference exact. Real metric noise is added on the metric's own scale (after g), and is often far larger (#7).
- **g only**: monotone increasing, one family (Kumaraswamy). There are no sign flips, failure plateaus or warps of x, and the function is the same (ρ = 0).
- **The BNN prior's near-zero noise** (sd ≈ 1e-3), so small errors in g cost nats (cf. notebook 02).
- **d ≤ 2.** n_B = 32·d is a choice, not calibrated.
- **The chain-rule likelihood** depends on the order of A's design; it is exact only for a Bayes-optimal PFN. The g posterior comes from importance sampling; its error is the MC check.


## Working hypotheses (project-level, 2026-10-22)

Beliefs that direct the project, stated so that experiments can confirm or refute them. Each is marked *claim* (to confirm) or *bias* (an inductive bias we intend to build into the prior or model).

**Terminology.**
- **The BNN prior** p(f) is the single-task prior. The frozen PFN approximates its posterior predictive.
- **The in-context prior** is the prediction for A at n_A = 0, given B alone. It is the related task's prediction, weighted by the pair prior's overall relatedness, including "unrelated".
- **The soft in-context prior.** As n_A grows, the prediction is the posterior p(f_A | D_B, D_A). Its likelihood must account for the registration (how A's x and y correspond to B's), so B acts as a soft prior rather than as data.

**H1 (claim): the prior is not yet representative of meta-HPO.** Relatedness through BNN weight perturbation (ρ) is neither calibrated nor interpretable. Starting from the BNN, relatedness should come from commonly accepted, intuitive patterns: shifted optima, rescaled outputs, local differences. These should be a few compositional primitives, not a hand-built simulator per distortion (catalogue: `docs/hpo_task_distortions.md`; #3).
*Tested by:* calibration against real benchmark pairs (later); per-primitive headroom experiments like this one.

**H2 (bias): what shifts between tasks is what x means.** The search space is shared (or can be made so through known parametrizations). If the optimum moves or an axis is transformed, the same x refers to a different place on the landscape. So the model must infer in context which (x, y) in B correspond to which in A: attention over (x, y)-similarity given the context, not x-distance. x-rank is ruled out too, because acquisition bias and small n_A make rankings unreliable.
*Tested by:* E1 architecture ablations (#2).

**H3 (claim): B gives a soft prior on where interesting points are, at most relatedness levels.** Even when B's values don't transfer exactly, B should tell a learner where A's good regions probably are.
*Tested by:* the n_A = 0 and small-n_A gains in every headroom experiment. Here: under a monotone output map, ranks transfer exactly, so H3 should hold most clearly. Later also in BO terms (regret), not only NLL.

**H4 (claim, this experiment): a low-dimensional output relation is identified from a few A points.** The verdict IDENTIFIED / PARTIAL / NONE above.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from meta_pfn.analysis import load_snapshot, show_header
from meta_pfn.analysis.outmap import gains, mc_error, verdict

SNAPSHOT = "results/outmap/<sweep_id>"  # set after `just snapshot runs/outmap/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

Nats per held-out A query; gains relative to `lower_bayes` (A alone, g inferred). CIs: 95% pair bootstrap. Members: see the header (DONE / failed / started / missing).

In [ ]:
cells = snap.runs.set_index("member")[["task.d", "task.kind"]].set_axis(["d", "kind"], axis=1)
def with_cell(name):
    return snap.table(name).join(cells, on="member")
nll, post = with_cell("nll"), with_cell("posterior")
g = gains(nll)
mc = mc_error(with_cell("mc"))
g.pivot_table(index=["d", "kind"], columns="n_mult", values=["gain_bayes", "gain_oracle", "G", "gain_naive", "gain_lower"]).round(3)

### Gain over A alone against n_A / d

Solid: Bayes (g inferred); dotted: g known; dashed: naive pooling. Shaded: 95% CI of the Bayes gain. n_A = 0 is the in-context prior.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True)
for (d, kind), h in g.groupby(["d", "kind"]):
    ax = axes[d - 1, ["identity", "outmap", "unrelated"].index(kind)]
    ax.plot(h.n_mult, h.gain_bayes, "-o", ms=3, c="C0", label="bayes")
    ax.fill_between(h.n_mult, h.gain_bayes_lo, h.gain_bayes_hi, color="C0", alpha=0.15)
    ax.plot(h.n_mult, h.gain_oracle, ":", c="k", label="g known")
    ax.plot(h.n_mult, h.gain_naive, "--", c="C1", label="naive")
    ax.axhline(0, c="k", lw=0.8); ax.set_xscale("symlog", linthresh=1); ax.set_xticks([0, 1, 2, 4, 8], [0, 1, 2, 4, 8])
    ax.set_title(f"d = {d}, {kind}"); ax.grid(alpha=0.3)
for ax in axes[-1]: ax.set_xlabel("n_A / d")
for ax in axes[:, 0]: ax.set_ylabel("gain over A alone (nats/query)")
axes[0, 0].legend(fontsize=8); fig.tight_layout()

### Diagnostics: posterior over kinds, g error, ESS, Monte-Carlo error

In [ ]:
display(post.groupby(["d", "kind", "n_a"])[["p_identity", "p_outmap", "p_unrelated"]].mean().round(2).unstack("n_a"))
display(post.groupby(["d", "kind", "n_a"])[["g_err", "ess_outmap"]].median().round(3).unstack("n_a"))
mc.pivot_table(index=["d", "kind"], columns="n_a", values="mc_error").round(3)

## Observations

_Filled from the outputs above._

In [ ]:
res = verdict(g, mc)
res

## Verdict

The criterion from the description, evaluated, in one line. Record it with `just decide`.